# Phase 2C.1 - Retrieval Screening (Colab)

Compare C0 recursive, C1 sentence-aware, C2 paragraph-aware, and C3 hierarchical chunking on the frozen 281-question development split. Retrieval is fixed to BGE-M3 sparse (`top_k=20`) plus BGE-large reranking (`top_n=5`). No generator or LLM judge is called.

Runs are sequential so latency remains comparable and the two large models do not compete for GPU memory. Start with `RUN_MODE='smoke'`; use `screening` only after the five-question run completes.

## 1. Configuration

In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='0f3586684deb95d310281754405a1773921b12f2'

C0_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
C0_REVISION='bb73e682f472933c212f2c6a3f9575c652b280fd'
C0_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
C0_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'

PHASE2C_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2c-indexes-v1'
PHASE2C_REVISION='09421ba33e75f2cef01dd01451ce4523214589ca'
PHASE2C_FILENAME='phase2c_chunking_indexes_v1.zip'
PHASE2C_SHA256='96030993b38de4d62828ce3d9945ae8fc73647c7c8fd0155d6eb223d0e6a0c90'

PREPARATION_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-experiments'
PREPARATION_REVISION='771fee6e19e3e836c9ba0b15874fe0bd5199b2a5'
PREPARATION_FILENAME='phase2b/preparation-refined-prompts-v1/phase2b_preparation_bundle.zip'
PREPARATION_SHA256='0b2ab8388d6ab679970519bc35ba574936a063b32406ce5cee4b6ec5c166073c'

RUN_MODE='smoke'       # smoke | screening
EXECUTE=False          # Set True after the preflight cells pass.
RUN_ARMS=('c3_hierarchical',)  # Use 'all' to rebuild C0-C3.
SEED=42
TOP_K=20
RERANK_TOP_N=5
C3_RERANK_CANDIDATE_N=TOP_K
RERANKER_MODEL='BAAI/bge-reranker-large'
EXPECTED_DEVELOPMENT_QUESTIONS=281
EXPECTED_DEVELOPMENT_ARTICLES=50
BOOTSTRAP_REPLICATES=2000
QUALITY_MARGIN=0.02
DEVICE='cuda:0'
RUN_VERSION='parent_backfill_v2'

COLAB_ROOT=Path('/content')
PROJECT_ROOT=COLAB_ROOT/'Text-Mining---NewsQA-RAG'
WORK_ROOT=COLAB_ROOT/f'newsqa_phase2c_retrieval_{RUN_MODE}_{RUN_VERSION}'
DATA_ROOT=WORK_ROOT/'data'
RUNTIME_ROOT=WORK_ROOT/'runtime'
RUNS_ROOT=WORK_ROOT/'runs'
RESULTS_ROOT=WORK_ROOT/'results'
LOG_ROOT=WORK_ROOT/'logs'
DRIVE_ROOT=COLAB_ROOT/'drive/MyDrive/newsqa_phase2c'
PRIOR_RESULTS_PATH=DRIVE_ROOT/f'phase2c_retrieval_{RUN_MODE}_results.zip'

## 2. Environment setup

In [ ]:
import hashlib,importlib.metadata as metadata,json,os,shutil,subprocess,sys,time,zipfile
from datetime import datetime,timezone
from google.colab import drive,userdata
from packaging.version import Version
drive.mount('/content/drive')
for path in [DATA_ROOT,RUNTIME_ROOT,RUNS_ROOT,RESULTS_ROOT,LOG_ROOT,DRIVE_ROOT]: path.mkdir(parents=True,exist_ok=True)
CHECKPOINT=DRIVE_ROOT/f'phase2c_retrieval_{RUN_MODE}_{RUN_VERSION}_checkpoint.zip'
if CHECKPOINT.exists() and not any(RUNS_ROOT.iterdir()):
    with zipfile.ZipFile(CHECKPOINT) as archive: archive.extractall(WORK_ROOT)
    print('Restored checkpoint:',CHECKPOINT)
assert RUN_MODE in {'smoke','screening'}
assert not REPO_COMMIT.startswith('SET_TO_'),'Pin REPO_COMMIT after committing this notebook'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','FlagEmbedding>=1.4.2'],check=True)
import numpy as np,pandas as pd,torch,yaml
import matplotlib.pyplot as plt
from IPython.display import display
assert Version(metadata.version('FlagEmbedding'))>=Version('1.4.2')
assert torch.cuda.is_available(),'Enable a Colab GPU runtime'
assert torch.cuda.get_device_capability(0)>=(7,0),'Use T4, L4 or A100; the installed PyTorch does not support P100'
try: HF_TOKEN=userdata.get('HF_TOKEN') or ''
except Exception: HF_TOKEN=''
assert HF_TOKEN,'Add the read token HF_TOKEN to Colab Secrets for the private C0 and preparation repositories'
os.environ.update({'CUDA_VISIBLE_DEVICES':'0','HF_HOME':str(COLAB_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','PYTHONPATH':str(PROJECT_ROOT/'common')})
print('GPU:',torch.cuda.get_device_name(0))
print('Repository commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())

## 3. Download and verify C0-C3 artifacts

In [ ]:
from huggingface_hub import hf_hub_download
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    return digest.hexdigest()
def extract_once(archive,target,marker):
    target.mkdir(parents=True,exist_ok=True)
    if not (target/marker).exists():
        with zipfile.ZipFile(archive) as source: source.extractall(target)
    assert (target/marker).exists(),f'Missing {marker} after extraction'
c0_zip=Path(hf_hub_download(repo_id=C0_REPO_ID,repo_type='dataset',revision=C0_REVISION,filename=C0_FILENAME,token=HF_TOKEN))
p2c_zip=Path(hf_hub_download(repo_id=PHASE2C_REPO_ID,repo_type='dataset',revision=PHASE2C_REVISION,filename=PHASE2C_FILENAME))
prep_zip=Path(hf_hub_download(repo_id=PREPARATION_REPO_ID,repo_type='dataset',revision=PREPARATION_REVISION,filename=PREPARATION_FILENAME,token=HF_TOKEN))
assert sha256_file(c0_zip)==C0_SHA256
assert sha256_file(p2c_zip)==PHASE2C_SHA256
assert sha256_file(prep_zip)==PREPARATION_SHA256
C0_ROOT=DATA_ROOT/'c0_recursive'; PHASE2C_ROOT=DATA_ROOT/'phase2c'; PREP_ROOT=DATA_ROOT/'preparation'
extract_once(c0_zip,C0_ROOT,'bundle_manifest.json')
extract_once(p2c_zip,PHASE2C_ROOT,'sentence/artifact_manifest.json')
extract_once(prep_zip,PREP_ROOT,'question_ids/development.json')
development_ids=json.loads((PREP_ROOT/'question_ids/development.json').read_text(encoding='utf-8'))
smoke_ids=json.loads((PREP_ROOT/'question_ids/smoke.json').read_text(encoding='utf-8'))
assert len(development_ids)==EXPECTED_DEVELOPMENT_QUESTIONS and len(set(development_ids))==len(development_ids)
assert len(smoke_ids)==5 and set(smoke_ids)<=set(development_ids)
active_ids=smoke_ids if RUN_MODE=='smoke' else development_ids
ACTIVE_IDS_PATH=RUNTIME_ROOT/f'{RUN_MODE}_question_ids.json'
ACTIVE_IDS_PATH.write_text(json.dumps(active_ids,indent=2)+'\n',encoding='utf-8')
print('Mode:',RUN_MODE,'| questions:',len(active_ids),'| artifacts verified')

## 4. Build portable runtime manifests

The published variants preserve their original build paths. This cell binds every variant to the extracted Colab paths and recomputes its configuration fingerprint.

In [ ]:
def load_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def jsonl_rows(path):
    with Path(path).open(encoding='utf-8') as handle: return [json.loads(line) for line in handle if line.strip()]
def stable_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
arms={
 'c0_recursive':{'root':C0_ROOT,'chunks':C0_ROOT/'chunks.jsonl','testset':C0_ROOT/'testset_resolved.jsonl','index':C0_ROOT/'bge_m3_sparse.pkl','variant':C0_ROOT/'deduplication/deduplicated.variant.json','chunking':{'strategy':'recursive','chunk_size':512,'chunk_overlap':64}},
 'c1_sentence':{'root':PHASE2C_ROOT/'sentence','chunks':PHASE2C_ROOT/'sentence/chunks.jsonl','testset':PHASE2C_ROOT/'sentence/testset_resolved.jsonl','index':PHASE2C_ROOT/'sentence/bge_m3_sparse.pkl','variant':PHASE2C_ROOT/'sentence/variant.json','chunking':{'strategy':'sentence','chunk_size':512,'chunk_overlap':64}},
 'c2_paragraph':{'root':PHASE2C_ROOT/'paragraph','chunks':PHASE2C_ROOT/'paragraph/chunks.jsonl','testset':PHASE2C_ROOT/'paragraph/testset_resolved.jsonl','index':PHASE2C_ROOT/'paragraph/bge_m3_sparse.pkl','variant':PHASE2C_ROOT/'paragraph/variant.json','chunking':{'strategy':'paragraph','chunk_size':512,'chunk_overlap':64}},
 'c3_hierarchical':{'root':PHASE2C_ROOT/'hierarchical','chunks':PHASE2C_ROOT/'hierarchical/chunks.jsonl','testset':PHASE2C_ROOT/'hierarchical/testset_resolved.jsonl','index':PHASE2C_ROOT/'hierarchical/bge_m3_sparse.pkl','variant':PHASE2C_ROOT/'hierarchical/variant.json','chunking':{'strategy':'hierarchical','chunk_size':512,'chunk_overlap':64,'child_chunk_size':256,'child_chunk_overlap':32}},
}
all_question_contracts={}
for arm,record in arms.items():
    for key in ['chunks','testset','index','variant']: assert Path(record[key]).exists(),f'{arm}: missing {key}'
    rows=jsonl_rows(record['testset']); by_id={row['question_id']:row for row in rows}
    assert set(active_ids)<=set(by_id),f'{arm}: active question IDs missing'
    contracts={qid:(by_id[qid]['question'],tuple(by_id[qid].get('accepted_answers',[])),by_id[qid].get('article_key')) for qid in active_ids}
    all_question_contracts[arm]=contracts
base_contract=all_question_contracts['c0_recursive']
assert all(value==base_contract for value in all_question_contracts.values()),'Question wording/answers/articles differ across arms'
active_articles={article for _,_,article in base_contract.values()}
if RUN_MODE=='screening': assert len(active_articles)==EXPECTED_DEVELOPMENT_ARTICLES
else: assert 1<=len(active_articles)<=len(active_ids)
for arm,record in arms.items():
    runtime=RUNTIME_ROOT/arm; runtime.mkdir(parents=True,exist_ok=True)
    config=yaml.safe_load((record['root']/'config.yaml').read_text()) if (record['root']/'config.yaml').exists() else yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
    config['chunking']=record['chunking']
    config.setdefault('retrieval',{})['retriever']='sparse'; config['retrieval']['top_k']=TOP_K
    config['retrieval'].setdefault('sparse',{}).update({'method':'bge-m3','model':'BAAI/bge-m3','model_name':'BAAI/bge-m3','device':DEVICE})
    config['retrieval'].setdefault('reranker',{}).update({'enabled':True,'type':'cross-encoder','model':RERANKER_MODEL,'top_n':RERANK_TOP_N,'batch_size':8,'device':DEVICE})
    config_path=runtime/'config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    variant=load_json(record['variant']); chunks_hash=sha256_file(record['chunks']); testset_hash=sha256_file(record['testset']); index_hash=sha256_file(record['index'])
    chunk_count=sum(1 for line in record['chunks'].open(encoding='utf-8') if line.strip())
    variant.setdefault('pipeline',{}).update({'config_path':str(config_path),'config_sha256':stable_hash(config),'chunking':record['chunking']})
    variant.setdefault('database',{}).update({'indexed':False,'chunk_count':chunk_count})
    artifacts=variant.setdefault('artifacts',{})
    artifacts['chunks']={'path':str(record['chunks']),'bytes':record['chunks'].stat().st_size,'sha256':chunks_hash}
    artifacts['testset_resolved']={'path':str(record['testset']),'bytes':record['testset'].stat().st_size,'sha256':testset_hash}
    artifacts['bm25']={'path':str(record['index']),'bytes':record['index'].stat().st_size,'sha256':index_hash}
    variant_path=runtime/'variant.json'; variant_path.write_text(json.dumps(variant,indent=2,sort_keys=True)+'\n',encoding='utf-8')
    record.update({'config':config_path,'runtime_variant':variant_path,'chunk_count':chunk_count,'chunks_sha256':chunks_hash,'testset_sha256':testset_hash,'index_sha256':index_hash})
display(pd.DataFrame([{'arm':arm,'chunks':r['chunk_count'],'index_mib':round(r['index'].stat().st_size/2**20,1)} for arm,r in arms.items()]))

## 5. Preflight and model cache

This verifies every mapped gold chunk and downloads the two fixed models before timed runs. Model download time is therefore excluded from retrieval latency.

In [ ]:
for arm,record in arms.items():
    chunk_ids={row['id'] for row in jsonl_rows(record['chunks'])}
    selected=[row for row in jsonl_rows(record['testset']) if row['question_id'] in set(active_ids)]
    missing={cid for row in selected for cid in row['relevant_chunk_ids'] if cid not in chunk_ids}
    assert len(selected)==len(active_ids) and not missing,f'{arm}: invalid gold mapping'
    if arm=='c3_hierarchical':
        links=jsonl_rows(record['root']/'child_parent_map.jsonl'); parents=jsonl_rows(record['root']/'parents.jsonl')
        child_parent={row['child_id']:row['parent_id'] for row in links}; parent_ids={row['id'] for row in parents}
        assert set(chunk_ids)==set(child_parent) and set(child_parent.values())<=parent_ids
from huggingface_hub import snapshot_download
snapshot_download('BAAI/bge-m3',token=HF_TOKEN or None)
snapshot_download(RERANKER_MODEL,token=HF_TOKEN or None)
print('Preflight passed. Set EXECUTE=True to run',RUN_MODE)

## 6. Sequential retrieval runs

In [ ]:
def run_command(command,label):
    log_path=LOG_ROOT/f'{label}.log'; print('$',' '.join(map(str,command)),flush=True)
    with log_path.open('a',encoding='utf-8') as log:
        process=subprocess.Popen(list(map(str,command)),cwd=PROJECT_ROOT,env={**os.environ,'PYTHONUNBUFFERED':'1'},stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
def save_checkpoint():
    local=Path(shutil.make_archive(str(COLAB_ROOT/f'phase2c_retrieval_{RUN_MODE}_{RUN_VERSION}_checkpoint'),'zip',root_dir=WORK_ROOT,base_dir='runs'))
    temporary=CHECKPOINT.with_suffix('.zip.tmp'); shutil.copy2(local,temporary); temporary.replace(CHECKPOINT)
    print('Checkpoint:',CHECKPOINT,f'{CHECKPOINT.stat().st_size/2**20:.1f} MiB')
assert EXECUTE,'Set EXECUTE=True after preflight'
selected_arms=list(arms) if RUN_ARMS=='all' else list(RUN_ARMS)
assert selected_arms and len(selected_arms)==len(set(selected_arms)) and set(selected_arms)<=set(arms),f'Invalid RUN_ARMS: {RUN_ARMS}'
reused_arms=[arm for arm in arms if arm not in selected_arms]; prior_results_sha256=None
if reused_arms:
    prior_zip=Path(PRIOR_RESULTS_PATH); assert prior_zip.exists(),f'Missing prior result bundle: {prior_zip}'
    prior_results_sha256=sha256_file(prior_zip); prior_root=WORK_ROOT/'prior_results'
    if not (prior_root/'run_manifest.json').exists():
        with zipfile.ZipFile(prior_zip) as archive: archive.extractall(prior_root)
    prior_manifest=load_json(prior_root/'run_manifest.json')
    assert prior_manifest['stage']=='phase2c_retrieval_screening' and prior_manifest['run_mode']==RUN_MODE
    assert prior_manifest['question_ids_sha256']==sha256_file(ACTIVE_IDS_PATH) and prior_manifest['questions']==len(active_ids),'Prior bundle uses a different question split'
    for arm in reused_arms:
        prior_arm=prior_manifest['arms'][arm]; current=arms[arm]
        assert prior_arm['chunks_sha256']==current['chunks_sha256'] and prior_arm['testset_sha256']==current['testset_sha256'] and prior_arm['index_sha256']==current['index_sha256'],f'{arm}: prior artifact hashes differ'
        current['run_dir']=prior_root/'runs'/arm
        report=load_json(current['run_dir']/'report.json'); assert report['coverage']['successful']==len(active_ids) and report['coverage']['success_rate']==1.0,f'{arm}: prior run is incomplete'
    print('Reusing prior arms:',reused_arms,'from',prior_zip)
for arm in selected_arms:
    record=arms[arm]
    run_dir=RUNS_ROOT/f'{RUN_MODE}_{arm}'; record['run_dir']=run_dir
    command=[sys.executable,'-u','scripts/collect_benchmark_predictions.py','--retriever','sparse','--reranker','cross-encoder','--reranker-model',RERANKER_MODEL,'--testset',record['testset'],'--variant-manifest',record['runtime_variant'],'--config',record['config'],'--run-dir',run_dir,'--question-ids-file',ACTIVE_IDS_PATH,'--chunks-path',record['chunks'],'--bm25-path',record['index'],'--top-k',TOP_K,'--rerank-top-n',RERANK_TOP_N,'--seed',SEED,'--retrieval-only','--max-attempts',3,'--retry-failed','--progress']
    if arm=='c3_hierarchical': command.extend(['--rerank-candidate-n',C3_RERANK_CANDIDATE_N])
    run_command(command,f'{RUN_MODE}_{arm}_collect')
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',run_dir],f'{RUN_MODE}_{arm}_score')
    report=load_json(run_dir/'report.json'); assert report['coverage']['successful']==len(active_ids) and report['coverage']['success_rate']==1.0
    save_checkpoint()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Executed arms:',selected_arms,'| reused arms:',reused_arms)

## 7. Compare metrics and C3 delivered parents

Standard C3 scores use retrieved child IDs. The additional delivered-parent score maps and deduplicates children to the parent contexts that a hierarchical generator would receive.

In [ ]:
sys.path.insert(0,str(PROJECT_ROOT/'common'))
from newsqa_rag.evaluation.benchmark_io import latest_by_question,load_jsonl
from newsqa_rag.evaluation.metrics import evaluate_retrieval
from newsqa_rag.retrieval.hierarchical import expand_ranked_children_to_parents
def nested(value,path,default=None):
    for key in path.split('.'): value=value.get(key,{})
    return default if value=={} else value
summary=[]
for arm,record in arms.items():
    report=load_json(record['run_dir']/'report.json')
    summary.append({'arm':arm,'questions':report['coverage']['successful'],'chunks':record['chunk_count'],'index_mib':round(record['index'].stat().st_size/2**20,2),'initial_hit@5':nested(report,'retrieval_initial.hit_rate@5'),'initial_recall@5':nested(report,'retrieval_initial.recall@5'),'hit@1':nested(report,'retrieval.hit_rate@1'),'hit@3':nested(report,'retrieval.hit_rate@3'),'hit@5':nested(report,'retrieval.hit_rate@5'),'recall@5':nested(report,'retrieval.recall@5'),'mrr@5':nested(report,'retrieval.mrr@5'),'ndcg@5':nested(report,'retrieval.ndcg@5'),'retrieve_p50_ms':nested(report,'latency.retrieve_ms.p50_ms'),'rerank_p50_ms':nested(report,'latency.rerank_ms.p50_ms'),'total_p50_ms':nested(report,'latency.total.p50_ms'),'total_p95_ms':nested(report,'latency.total.p95_ms')})
comparison=pd.DataFrame(summary)
c3=arms['c3_hierarchical']; child_parent={row['child_id']:row['parent_id'] for row in jsonl_rows(c3['root']/'child_parent_map.jsonl')}; parent_by_id={row['id']:row for row in jsonl_rows(c3['root']/'parents.jsonl')}
c3_records=latest_by_question(load_jsonl(c3['run_dir']/'predictions.jsonl'))
parent_samples=[]; parent_scores={}; parent_counts={}
for qid in active_ids:
    record=c3_records[qid]; trace=record.get('result') or record.get('retrieval_trace') or {}
    gold=list(dict.fromkeys(child_parent[cid] for cid in record['relevant_chunk_ids']))
    delivered=[parent['id'] for parent in expand_ranked_children_to_parents(trace.get('reranked_chunks',[]),child_parent,parent_by_id,RERANK_TOP_N)]
    parent_counts[qid]=len(delivered)
    sample={'relevant_chunk_ids':gold,'retrieved_ids':delivered}; parent_samples.append(sample)
    parent_scores[qid]=evaluate_retrieval([sample],[1,3,5])
short_parent_sets={qid:count for qid,count in parent_counts.items() if count<RERANK_TOP_N}
assert not short_parent_sets,f'C3 failed to backfill five unique parents: {list(short_parent_sets.items())[:5]}'
parent_metrics=evaluate_retrieval(parent_samples,[1,3,5])
with (RESULTS_ROOT/f'c3_parent_scores_{RUN_MODE}.jsonl').open('w',encoding='utf-8') as handle:
    for qid in active_ids: handle.write(json.dumps({'question_id':qid,'article_key':c3_records[qid]['article_key'],'delivered_parent_count':parent_counts[qid],'retrieval':parent_scores[qid]})+'\n')
for key in ['hit_rate@5','recall@5','mrr@5','ndcg@5']: comparison.loc[comparison.arm=='c3_hierarchical',f'parent_{key}']=parent_metrics[key]
comparison.to_csv(RESULTS_ROOT/f'phase2c_retrieval_{RUN_MODE}.csv',index=False)
display(comparison.sort_values('hit@5',ascending=False))
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
comparison.set_index('arm')[['hit@5','recall@5','mrr@5','ndcg@5']].plot.bar(ax=axes[0],ylim=(0,1),title='Post-rerank retrieval quality')
comparison.set_index('arm')[['retrieve_p50_ms','rerank_p50_ms','total_p50_ms']].plot.bar(ax=axes[1],title='Median latency (ms)')
for axis in axes: axis.grid(axis='y',alpha=.25); axis.tick_params(axis='x',rotation=25)
plt.tight_layout(); plt.savefig(RESULTS_ROOT/f'phase2c_retrieval_{RUN_MODE}.png',dpi=160,bbox_inches='tight'); plt.show()

## 8. Paired article-cluster bootstrap and eligibility

The automatic rule only rejects an arm when Hit@5 or Recall@5 is more than 0.02 below C0 and the 95% cluster-bootstrap interval excludes zero. Smoke results are diagnostic and never select an arm.

In [ ]:
score_rows={}
for arm,record in arms.items(): score_rows[arm]={row['question_id']:row for row in load_jsonl(record['run_dir']/'deterministic_scores.jsonl')}
assert all(set(rows)==set(active_ids) for rows in score_rows.values())
metric_paths={'hit@5':'retrieval.hit_rate@5','recall@5':'retrieval.recall@5','mrr@5':'retrieval.mrr@5','ndcg@5':'retrieval.ndcg@5'}
def metric_value(arm,qid,metric,path):
    if arm=='c3_hierarchical': return float(parent_scores[qid][metric.replace('hit@5','hit_rate@5')])
    return float(nested(score_rows[arm][qid],path,0.0))
article_groups={}
for qid,row in score_rows['c0_recursive'].items(): article_groups.setdefault(row['article_key'],[]).append(qid)
rng=np.random.default_rng(SEED); articles=sorted(article_groups); bootstrap={}
for arm in arms:
    bootstrap[arm]={}
    for metric,path in metric_paths.items():
        deltas={article:np.array([metric_value(arm,qid,metric,path)-metric_value('c0_recursive',qid,metric,path) for qid in qids],dtype=float) for article,qids in article_groups.items()}
        observed=float(np.mean(np.concatenate(list(deltas.values())))); samples=[]
        for _ in range(BOOTSTRAP_REPLICATES):
            sampled=rng.choice(articles,size=len(articles),replace=True); samples.append(float(np.mean(np.concatenate([deltas[a] for a in sampled]))))
        bootstrap[arm][metric]={'delta_vs_c0':round(observed,6),'ci95_low':round(float(np.percentile(samples,2.5)),6),'ci95_high':round(float(np.percentile(samples,97.5)),6)}
eligibility={}
for arm in arms:
    reasons=[]
    if len(score_rows[arm])!=len(active_ids): reasons.append('coverage_below_100_percent')
    for metric in ['hit@5','recall@5']:
        result=bootstrap[arm][metric]
        if result['delta_vs_c0'] < -QUALITY_MARGIN and result['ci95_high'] < 0: reasons.append(f'{metric}_drop_exceeds_margin_with_negative_ci')
    eligibility[arm]={'eligible':not reasons,'reasons':reasons}
decision={'schema_version':1,'run_mode':RUN_MODE,'selection_permitted':RUN_MODE=='screening','control':'c0_recursive','c3_primary_unit':'deduplicated_delivered_parent','c3_rerank_candidate_n':C3_RERANK_CANDIDATE_N,'c3_delivered_parent_n':RERANK_TOP_N,'c3_parent_count_distribution':{str(count):list(parent_counts.values()).count(count) for count in sorted(set(parent_counts.values()))},'quality_margin':QUALITY_MARGIN,'bootstrap_replicates':BOOTSTRAP_REPLICATES,'bootstrap_unit':'article','bootstrap':bootstrap,'eligibility':eligibility}
(RESULTS_ROOT/f'phase2c_retrieval_{RUN_MODE}_eligibility.json').write_text(json.dumps(decision,indent=2,sort_keys=True)+'\n',encoding='utf-8')
display(pd.DataFrame([{'arm':arm,'eligible':value['eligible'],'reasons':'; '.join(value['reasons']),'hit_delta':bootstrap[arm]['hit@5']['delta_vs_c0'],'hit_ci':f"[{bootstrap[arm]['hit@5']['ci95_low']:.4f}, {bootstrap[arm]['hit@5']['ci95_high']:.4f}]",'recall_delta':bootstrap[arm]['recall@5']['delta_vs_c0'],'recall_ci':f"[{bootstrap[arm]['recall@5']['ci95_low']:.4f}, {bootstrap[arm]['recall@5']['ci95_high']:.4f}]"} for arm,value in eligibility.items()]))

## 9. Provenance and downloadable result bundle

In [ ]:
manifest={'schema_version':1,'stage':'phase2c_retrieval_screening','run_mode':RUN_MODE,'run_version':RUN_VERSION,'created_at':datetime.now(timezone.utc).isoformat(),'repo_commit':REPO_COMMIT,'question_ids_sha256':sha256_file(ACTIVE_IDS_PATH),'questions':len(active_ids),'articles':len(article_groups),'execution':{'executed_arms':selected_arms,'reused_arms':reused_arms,'prior_results_path':str(PRIOR_RESULTS_PATH) if reused_arms else None,'prior_results_sha256':prior_results_sha256},'retrieval':{'method':'bge-m3','top_k':TOP_K,'reranker':RERANKER_MODEL,'rerank_top_n':RERANK_TOP_N,'rerank_candidate_n':{'default':RERANK_TOP_N,'c3_hierarchical':C3_RERANK_CANDIDATE_N},'execution':'sequential'},'sources':{'c0':{'repo_id':C0_REPO_ID,'revision':C0_REVISION,'filename':C0_FILENAME,'sha256':C0_SHA256},'phase2c':{'repo_id':PHASE2C_REPO_ID,'revision':PHASE2C_REVISION,'filename':PHASE2C_FILENAME,'sha256':PHASE2C_SHA256},'partition':{'repo_id':PREPARATION_REPO_ID,'revision':PREPARATION_REVISION,'filename':PREPARATION_FILENAME,'sha256':PREPARATION_SHA256}},'arms':{arm:{key:str(value) if isinstance(value,Path) else value for key,value in record.items() if key in {'chunk_count','chunks_sha256','testset_sha256','index_sha256'}} for arm,record in arms.items()}}
(RESULTS_ROOT/'run_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
for arm,record in arms.items():
    destination=RESULTS_ROOT/'runs'/arm; destination.mkdir(parents=True,exist_ok=True)
    for name in ['run_manifest.json','report.json','report_summary.txt','deterministic_scores.jsonl','retrievals.jsonl','predictions.jsonl','environment.json']:
        source=record['run_dir']/name
        if source.exists(): shutil.copy2(source,destination/name)
bundle=Path(shutil.make_archive(str(WORK_ROOT/f'phase2c_retrieval_{RUN_MODE}_results'),'zip',root_dir=RESULTS_ROOT))
destination=DRIVE_ROOT/bundle.name; temporary=destination.with_suffix('.zip.tmp'); shutil.copy2(bundle,temporary); temporary.replace(destination)
print('Result bundle:',destination,f'{destination.stat().st_size/2**20:.1f} MiB')
print('SHA-256:',sha256_file(destination))
print('Next: inspect the CSV and eligibility JSON. Do not use smoke output for selection.')